Import Libraries

In [1]:
import jax
import flax
import optax
from jax import lax, random, numpy as jnp
from jax import random, grad, vmap, hessian, jacfwd, jit
from jax import config
from flax import linen as nn
from evojax.util import get_params_format_fn

import time
import numpy as np
import pandas as pd
from scipy import io
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize
from scipy.optimize import minimize_scalar


from matplotlib import rcParams
import math
# choose GPU
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
#jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_matmul_precision", "highest")

# config = {
#     "font.family": 'Times New Roman',
#     "font.size": 14,
#     "mathtext.fontset": 'stix',
#     "font.serif": ['SimSun'],
# }
# rcParams.update(config)

# rcParams['axes.unicode_minus'] = False

Problem: Navier-Stokes Equation

        u*u_x + v*u_y - 1/Re*(u_xx+u_yy) + p_x = 0
        v*v_x + v*v_y - 1/Re*(v_xx+v_yy) + p_y = 0
        u_x + v_y = 0

In [2]:
# parameter
Re = 100
afa_ref = 0.2 



In [3]:
sim = pd.read_csv('wavy_channel_361x73_Re100.csv',sep=',')
print("x: ", sim['x'].shape, ", y: " , sim['y'].shape,", u: ", sim['u'].shape, ", v: " , sim['v'].shape, "p: ", sim['p'].shape)

x:  (26353,) , y:  (26353,) , u:  (26353,) , v:  (26353,) p:  (26353,)


In [4]:
sim

,x,y,u,v,p,ls
0,0.00,-1.8,0.000000e+00,0.000000e+00,0.000000,0.8
1,0.05,-1.8,-4.927610e-09,1.239922e-10,-0.003375,0.8
2,0.10,-1.8,8.962268e-09,-2.063832e-10,-0.013445,0.8
3,0.15,-1.8,-2.973871e-09,5.015437e-11,0.005333,0.8
4,0.20,-1.8,1.063466e-09,-2.250247e-11,-0.002667,0.8
...,...,...,...,...,...,...
26348,17.80,1.8,NaN,NaN,NaN,NaN
26349,17.85,1.8,NaN,NaN,NaN,NaN
26350,17.90,1.8,NaN,NaN,NaN,NaN
26351,17.95,1.8,NaN,NaN,NaN,NaN


In [5]:
sim_x = sim['x'].values.reshape(-1,1)
sim_y = sim['y'].values.reshape(-1,1)
# sim_ls = sim['ls'].values.reshape(-1,1)
sim_u = sim['u'].values.reshape(-1,1)
sim_v = sim['v'].values.reshape(-1,1)
sim_p = sim['p'].values.reshape(-1,1)

x_unique, y_unique = np.unique(sim_x), np.unique(sim_y)

dx = 0.05
dy = 0.05
print ('dx:',dx, 'dy:',dy)
print ( 'x_unique shape: ', x_unique.shape)
print ( 'y_unique shape: ', y_unique.shape)

dx: 0.05 dy: 0.05
x_unique shape:  (361,)
y_unique shape:  (73,)


In [6]:
x_l, x_u, y_l, y_u = np.min(sim_x), np.max(sim_x), np.min(sim_y), np.max(sim_y)

ext = [x_l, x_u, y_l, y_u]
print (x_l, x_u, y_l, y_u)


0.0 18.0 -1.8 1.8000000000000032


In [7]:
def wavy(x):
    """Reference velocity profile"""
    return 1 + afa_ref * jnp.sin(jnp.pi * (x - 3))


In [8]:
data_X, data_Y = np.hstack([sim_x, sim_y]), np.hstack([sim_u, sim_v, sim_p])
data_X = np.round(data_X, 5)

In [9]:
def filter_data(data_X, data_Y):
    x = data_X[:, 0]
    y = data_X[:, 1]
    
    mask_left = (x > 0.0) & (x <= 3.0) & (y > -1 ) & (y < 1)
    mask_right = (x >= 15.0) & (x < 18.0) & (y > -1) & (y < 1)
    mask_middle = (x > 3.0) & (x < 15.0) & (y > -wavy(x)) & (y < wavy(x))
    
    mask = mask_left | mask_right | mask_middle
    return data_X[mask], data_Y[mask]
data_X_inter, data_Y_inter = filter_data(data_X, data_Y)


In [10]:
l_line_bool = (data_X[:, 0] <= 3)
r_line_bool = (data_X[:, 0] >= 15)
x_line = np.unique(data_X[l_line_bool | r_line_bool][:, 0]).reshape(-1, 1)
y_line_t = np.ones_like(x_line) * 1.0
y_line_b = np.ones_like(x_line) * -1.0
x_lines = np.vstack([x_line, x_line])
y_lines = np.vstack([y_line_t, y_line_b])

data_X_bc_line = np.hstack([x_lines, y_lines])
data_Y_bc_line = np.zeros((x_lines.shape[0], 3))  # u, v, p


In [11]:
lbc_bool = (np.isclose(data_X[:, 0], x_l)) & (data_X[:, 1] >= -1) & (data_X[:, 1] <= 1 )
data_X_lbc = data_X[lbc_bool]
data_Y_lbc = data_Y[lbc_bool]

rbc_bool = (np.isclose(data_X[:, 0], x_u)) & (data_X[:, 1] >= -1) & (data_X[:, 1] <= 1 )
data_X_rbc = data_X[rbc_bool]
data_Y_rbc = data_Y[rbc_bool]


wavy_x_bool = ((3 < data_X[:, 0]) & (data_X[:, 0] < 15))
data_x_wavy_ =  np.unique(data_X[wavy_x_bool][:,0])
data_y_wavy_up = wavy(data_x_wavy_)
data_y_wavy_down = -wavy(data_x_wavy_)
data_x_wavy = jnp.hstack([data_x_wavy_, data_x_wavy_]).reshape(-1, 1)
data_y_wavy = jnp.hstack([data_y_wavy_up, data_y_wavy_down]).reshape(-1, 1)
data_X_wavy = jnp.hstack([data_x_wavy, data_y_wavy])
data_Y_wavy =  np.zeros((data_X_wavy.shape[0], 3))


data_X_bc = jnp.vstack([data_X_lbc, data_X_rbc, data_X_wavy, data_X_bc_line])
data_Y_bc = jnp.vstack([data_Y_lbc, data_Y_rbc, data_Y_wavy, data_Y_bc_line])
print("data_X_bc shape: ", data_X_bc.shape, ", data_Y_bc shape: ", data_Y_bc.shape)

data_X_bc shape:  (804, 2) , data_Y_bc shape:  (804, 3)


In [12]:
l_bool = (data_X_bc[:, 0] == x_l)
r_bool = (data_X_bc[:, 0] == x_u)
tb_bool = (~l_bool & ~r_bool)
data_X_bc_l = data_X_bc[l_bool]
data_X_bc_r = data_X_bc[r_bool]
data_Y_bc_l = data_Y_bc[l_bool]
data_Y_bc_r = data_Y_bc[r_bool]
data_X_bc_tb = data_X_bc[tb_bool]
data_Y_bc_tb = data_Y_bc[tb_bool]

print("data_X_bc shape: ", data_X_bc.shape, ", data_Y_bc shape: ", data_Y_bc.shape)
print("data_X_bc_l shape: ", data_X_bc_l.shape, ", data_Y_bc_l shape: ", data_Y_bc_l.shape)
print("data_X_bc_r shape: ", data_X_bc_r.shape, ", data_Y_bc_r shape: ", data_Y_bc_r.shape)
print("data_X_bc_tb shape: ", data_X_bc_tb.shape, ", data_Y_bc_tb shape: ", data_Y_bc_tb.shape)

data_X_bc shape:  (804, 2) , data_Y_bc shape:  (804, 3)
data_X_bc_l shape:  (43, 2) , data_Y_bc_l shape:  (43, 3)
data_X_bc_r shape:  (43, 2) , data_Y_bc_r shape:  (43, 3)
data_X_bc_tb shape:  (718, 2) , data_Y_bc_tb shape:  (718, 3)


In [13]:
bc_top_y= np.vstack([data_y_wavy_up.reshape(-1,1), y_line_t.reshape(-1,1)])
bc_top_x = np.vstack([data_x_wavy_.reshape(-1,1), x_line.reshape(-1,1)])
bc_top_x, bc_top_y = bc_top_x[np.argsort(bc_top_x[:, 0])], bc_top_y[np.argsort(bc_top_x[:, 0])]


In [14]:
print(data_X_inter.shape, data_Y_inter.shape)

(14225, 2) (14225, 3)


In [15]:
import numpy as np
from scipy.interpolate import interp1d

def select_valid_points(data_X, bc_top_x, bc_top_y, dx, dy):

    bc_x = bc_top_x.flatten()
    bc_y = bc_top_y.flatten()

    f_upper = interp1d(bc_x, bc_y, bounds_error=False, fill_value=np.nan)
    f_lower = interp1d(bc_x, -bc_y, bounds_error=False, fill_value=np.nan)

    neighbors = np.stack([
        data_X,  # self
        data_X + np.array([ dx,  0]),  # right
        data_X + np.array([-dx,  0]),  # left
        data_X + np.array([ 0,  dy]),  # up
        data_X + np.array([ 0, -dy]),  # down
    ], axis=1)  # shape: (N, 5, 2)

    flat_neighbors = neighbors.reshape(-1, 2)
    x_vals = flat_neighbors[:, 0]
    y_vals = flat_neighbors[:, 1]

    y_upper = f_upper(x_vals)
    y_lower = f_lower(x_vals)

    inside_mask_flat = (y_vals >= y_lower) & (y_vals <= y_upper) & \
                       ~np.isnan(y_upper) & ~np.isnan(y_lower)

    inside_mask = inside_mask_flat.reshape(-1, 5)

    valid_mask = np.all(inside_mask, axis=1)

    valid_points = data_X[valid_mask]

    return valid_points, valid_mask

train_X, in_channel_mask = select_valid_points(data_X_inter, bc_top_x, bc_top_y, dx=dx, dy=dy)
train_Y = data_Y_inter[in_channel_mask]
print("train_X shape: ", train_X.shape, ", train_Y shape: ", train_Y.shape)

train_X shape:  (13783, 2) , train_Y shape:  (13783, 3)


In [16]:
ad_points_X = data_X_inter[~in_channel_mask]
ad_points_Y = data_Y_inter[~in_channel_mask]
print("ad_points_X shape: ", ad_points_X.shape, ", ad_points_Y shape: ", ad_points_Y.shape)

ad_points_X shape:  (442, 2) , ad_points_Y shape:  (442, 3)


In [17]:
train_X, train_Y = jnp.array(train_X), jnp.array(train_Y)  
data_X_bc_tb, data_Y_bc_tb = jnp.array(data_X_bc_tb), jnp.array(data_Y_bc_tb)  
data_X_l_BC, data_Y_l_BC = jnp.array(data_X_bc_l), jnp.array(data_Y_bc_l) 
data_X_r_BC, data_Y_r_BC = jnp.array(data_X_bc_r), jnp.array(data_Y_bc_r) 
ad_points_X, ad_points_Y = jnp.array(ad_points_X), jnp.array(ad_points_Y) 

print("train_X shape: ", train_X.shape, ", train_Y shape: ", train_Y.shape)
print("data_X_bc_tb shape: ", data_X_bc_tb.shape, ", data_Y_bc_tb shape: ", data_Y_bc_tb.shape)
print("data_X_l_BC shape: ", data_X_l_BC.shape, ", data_Y_l_BC shape: ", data_Y_l_BC.shape)
print("data_X_r_BC shape: ", data_X_r_BC.shape, ", data_Y_r_BC shape: ", data_Y_r_BC.shape)
print("ad_points_X shape: ", ad_points_X.shape, ", ad_points_Y shape: ", ad_points_Y.shape)


train_X shape:  (13783, 2) , train_Y shape:  (13783, 3)
data_X_bc_tb shape:  (718, 2) , data_Y_bc_tb shape:  (718, 3)
data_X_l_BC shape:  (43, 2) , data_Y_l_BC shape:  (43, 3)
data_X_r_BC shape:  (43, 2) , data_Y_r_BC shape:  (43, 3)
ad_points_X shape:  (442, 2) , ad_points_Y shape:  (442, 3)


In [18]:
BS_PDE = int(len(train_X)*0.2)
BS_l_BC = int(len(data_X_l_BC)*0.5)
BS_r_BC = int(len(data_X_r_BC)*0.5)
BS_tb_BC = int(len(data_X_bc_tb)*0.1)
BS_ad = int(len(ad_points_X)*0.1)

BS_all = BS_PDE + BS_l_BC + BS_r_BC + BS_tb_BC + BS_ad

n_pde, n_l_BC, n_r_BC, n_tb_BC, n_ad  = len(train_X), len(data_X_l_BC), len(data_X_r_BC), len(data_X_bc_tb), len(ad_points_X)
n_all = n_pde + n_l_BC + n_r_BC + n_tb_BC + n_ad

print('The total number of samples:')
print('n_all: ',n_pde, 'n_l_BC: ',n_l_BC, 'n_r_BC: ',n_r_BC, 'n_tb_BC: ',n_tb_BC, 'n_ad: ',n_ad, 'n_all: ',n_all)
print('The number of samples in each batch:')
print('BS_ALL: ',BS_PDE, 'BS_l_BC: ',BS_l_BC, 'BS_r_BC: ',BS_r_BC, 'BS_tb_BC: ',BS_tb_BC, 'BS_ad: ',BS_ad, 'All points: ',BS_all)

The total number of samples:
n_all:  13783 n_l_BC:  43 n_r_BC:  43 n_tb_BC:  718 n_ad:  442 n_all:  15029
The number of samples in each batch:
BS_ALL:  2756 BS_l_BC:  21 BS_r_BC:  21 BS_tb_BC:  71 BS_ad:  44 All points:  2913


DNN / PINN   

In [19]:
nn_acf = nn.silu
class PINN(nn.Module):
    """PINNs"""
    def setup(self):
        # initialization
        kinit = jax.nn.initializers.he_uniform()
        # feature mapping later
        self.feats = nn.Dense(n_nodes * 2, kernel_init = kinit)
        # hidden layers
        self.layers = [nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf]
        # split layers
        self.splitu = nn.Dense(n_nodes, kernel_init = kinit)
        self.layeru = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitv = nn.Dense(n_nodes, kernel_init = kinit)
        self.layerv = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitp = nn.Dense(n_nodes, kernel_init = kinit)      
        self.layerp = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]     


    @nn.compact
    def __call__(self, inputs):
        # split the two variables, probably just by slicing
        x, y = inputs[:,0:1], inputs[:,1:2]
         
        def get_uvp(x, y):

            inputs = jnp.hstack([x,y])
            # feature mapping
            hidden = self.feats(inputs)
            hidden = jnp.sin(jnp.pi*hidden)
            # share hidden layer
            for i, lyr in enumerate(self.layers):
                hidden = lyr(hidden)
            # split hidden layer
            u = self.splitu(hidden)
            for i, lyr in enumerate(self.layeru):
                u = lyr(u)  
            v = self.splitv(hidden)
            for i, lyr in enumerate(self.layerv):
                v = lyr(v)   
            p = self.splitp(hidden)
            for i, lyr in enumerate(self.layerp):
                p = lyr(p) 

            return (u,v,p)  
    
        u, v, p = get_uvp(x, y)

        xE, xW = x + dx, x - dx
        yN, yS = y + dy, y - dy
        xe, xw = x + 0.5*dx, x - 0.5*dx
        yn, ys = y + 0.5*dy, y - 0.5*dy
        # obtain u, v, p neighbour
        uE, vE, pE = get_uvp(xE, y)
        uW, vW, pW = get_uvp(xW, y)
        uN, vN, pN = get_uvp(x, yN)
        uS, vS, pS = get_uvp(x, yS)
        ue, ve, pe = get_uvp(xe, y)
        uw, vw, pw = get_uvp(xw, y)
        un, vn, pn = get_uvp(x, yn)
        us, vs, ps = get_uvp(x, ys)

        ae = ue
        aw = -uw
        an = vn
        aas = -vs
        
        outlet_x = (x_u - dx)*jnp.ones_like(x)
        u_outlet,v_outlet,_ = get_uvp(outlet_x, y)
        outlet_bc_u = u_outlet - u
        outlet_bc_v = v_outlet - v

        
        source_x = (pe - pw)
        source_y = (pn - ps) 

        l_bc = (x == x_l)
        r_bc = (x == x_u)
        
        div = (uE - uW + vN - vS)/2


        mom_x = ae*ue + aw*uw + an*un + aas*us + source_x - (uE + uW + uN + uS - 4*u)/(dx*Re) 
        mom_y = ae*ve + aw*vw + an*vn + aas*vs + source_y - (vE + vW + vN + vS - 4*v)/(dx*Re) 

        res_u = ( source_x - (uE + uW + uN + uS)/(dx*Re))/dx
        res_v = ( source_y - (vE + vW + vN + vS)/(dx*Re))/dx 
        
        residuals_continuity = div/dx
        residuals_momentum_1 = mom_x/dx 
        residuals_momentum_2 = mom_y/dy 
        res_c = (ue - uw + vn - vs)/dx
        res_p = pE + pW + pN + pS
        
        outputs = jnp.hstack([u, v, p, residuals_continuity, residuals_momentum_1, residuals_momentum_2,res_u,res_v, outlet_bc_u,outlet_bc_v,l_bc, r_bc,res_c,res_p])

        return outputs    

  
    
class DNN(nn.Module):
    """DNNs"""
    def setup(self):
        # initialization
        kinit = jax.nn.initializers.he_uniform()
        # feature mapping later
        self.feats = nn.Dense(n_nodes * 2, kernel_init = kinit)
        # hidden layers
        self.layers = [nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf]
        # split layers
        self.splitu = nn.Dense(n_nodes, kernel_init = kinit)
        self.layeru = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitv = nn.Dense(n_nodes, kernel_init = kinit)
        self.layerv = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitp = nn.Dense(n_nodes, kernel_init = kinit)      
        self.layerp = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]    


    @nn.compact
    def __call__(self, inputs):
        # split the two variables, probably just by slicing
        x, y = inputs[:,0:1], inputs[:,1:2]

        def get_uvp(x, y):

            inputs = jnp.hstack([x,y])
            # feature mapping
            hidden = self.feats(inputs)
            # hidden = jnp.sin(hidden)
            hidden = jnp.sin(jnp.pi*hidden)

            # share hidden layer
            for i, lyr in enumerate(self.layers):
                hidden = lyr(hidden)
            # split hidden layer
            u = self.splitu(hidden)
            for i, lyr in enumerate(self.layeru):
                u = lyr(u)  
            v = self.splitv(hidden)
            for i, lyr in enumerate(self.layerv):
                v = lyr(v)   
            p = self.splitp(hidden)
            for i, lyr in enumerate(self.layerp):
                p = lyr(p)

            return (u,v,p)  
    
        u, v, p = get_uvp(x, y)

        xE, xW = x + dx, x - dx
        yN, yS = y + dy, y - dy
        xe, xw = x + 0.5*dx, x - 0.5*dx
        yn, ys = y + 0.5*dy, y - 0.5*dy
        
        # obtain u, v, p neighbour
        uE, vE, pE = get_uvp(xE, y)
        uW, vW, pW = get_uvp(xW, y)
        uN, vN, pN = get_uvp(x, yN)
        uS, vS, pS = get_uvp(x, yS)
        ue, ve, pe = get_uvp(xe, y)
        uw, vw, pw = get_uvp(xw, y)
        un, vn, pn = get_uvp(x, yn)
        us, vs, ps = get_uvp(x, ys)

        source_x = (pe - pw)
        source_y = (pn - ps)
      
        res_u = ( source_x - (uE + uW + uN + uS)/(dx*Re))/dx #-fx
        res_v = ( source_y - (vE + vW + vN + vS)/(dx*Re))/dx #-fy

        res_c = (ue - uw + vn - vs)/dx
        res_p = pE + pW + pN + pS
                
        outputs = jnp.hstack([u, v, p,res_u,res_v,res_c,res_p]) 
        return outputs    
    
class DNN_interpolation(nn.Module):
    """interploration DNNs"""
    def setup(self):
        # initialization
        kinit = jax.nn.initializers.he_uniform()
        # feature mapping later
        self.feats = nn.Dense(n_nodes * 2, kernel_init = kinit)
        # hidden layers
        self.layers = [nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf]
        # split layers
        self.splitu = nn.Dense(n_nodes, kernel_init = kinit)
        self.layeru = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitv = nn.Dense(n_nodes, kernel_init = kinit)
        self.layerv = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]
        self.splitp = nn.Dense(n_nodes, kernel_init = kinit)      
        self.layerp = [nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,
                       nn.Dense(n_nodes, kernel_init = kinit),
                       nn_acf,                       
                       nn.Dense(1, kernel_init = kinit, use_bias=False),
                       ]  



    @nn.compact
    def __call__(self, inputs):
        # split the two variables, probably just by slicing
        x, y = inputs[-BS_ad:,0:1], inputs[-BS_ad:,1:2]

        
        def get_uvp(x, y):

            inputs = jnp.hstack([x,y])
            # feature mapping
            hidden = self.feats(inputs)
            hidden = jnp.sin(jnp.pi*hidden)
            # hidden = jnp.sin(hidden)

            # share hidden layer
            for i, lyr in enumerate(self.layers):
                hidden = lyr(hidden)
            # split hidden layer
            u = self.splitu(hidden)
            for i, lyr in enumerate(self.layeru):
                u = lyr(u)  
            v = self.splitv(hidden)
            for i, lyr in enumerate(self.layerv):
                v = lyr(v)   
            p = self.splitp(hidden)
            for i, lyr in enumerate(self.layerp):
                p = lyr(p) 
 
            return (u,v,p)  
        
        u, v, p = get_uvp(x, y)
   
        # obtain u_x, u_y, v_x, v_y, p_x, p_y
        def get_uvp_xy(get_uvp, x, y):
            u_x, v_x, p_x = jacfwd(get_uvp)(x, y)
            u_y, v_y, p_y = jacfwd(get_uvp, argnums=1)(x, y)

            return u_x, u_y, v_x, v_y, p_x , p_y
            
        get_uvp_xy_vmap = vmap(get_uvp_xy, in_axes=(None, 0, 0))
        u_x, u_y, v_x, v_y, p_x , p_y = get_uvp_xy_vmap(get_uvp, x, y)
        u_x, u_y, v_x, v_y, p_x , p_y = u_x[:,:,0], u_y[:,:,0], v_x[:,:,0], v_y[:,:,0], p_x[:,:,0] , p_y[:,:,0]

       # obtain u_xx, u_yy, v_xx, v_yy, p_xx, p_yy
        def get_uvp_xxyy(get_uvp, x, y):
            u_xx, v_xx, p_xx = jacfwd(jacfwd(get_uvp))(x, y)
            u_yy, v_yy, p_yy = jacfwd(jacfwd(get_uvp, argnums=1), argnums=1)(x, y)

            return u_xx, u_yy, v_xx, v_yy, p_xx , p_yy
        
        get_uvp_xxyy_vmap = vmap(get_uvp_xxyy, in_axes=(None, 0, 0))
        u_xx, u_yy, v_xx, v_yy, p_xx , p_yy = get_uvp_xxyy_vmap(get_uvp, x, y)
        u_xx, u_yy, v_xx, v_yy, p_xx , p_yy = u_xx[:,:,0,0], u_yy[:,:,0,0], v_xx[:,:,0,0], v_yy[:,:,0,0], p_xx[:,:,0,0] , p_yy[:,:,0,0]

        ad_continuity = u_x + v_y
        ad_momentum_1 = u*u_x + v*u_y + p_x - 1.0/Re*(u_xx + u_yy)
        ad_momentum_2 = u*v_x + v*v_y + p_y - 1.0/Re*(v_xx + v_yy)

        
        outputs = jnp.hstack([ad_continuity,ad_momentum_1,ad_momentum_2])

        return outputs     

In [20]:
# choose seed
seed = 10
key, rng = random.split(random.PRNGKey(seed))

# dummy input
a = random.normal(key, [1,8])

# initialization call
bi_count = 1
n_nodes = 32
model, model_0,model_interp = PINN(), DNN(),DNN_interpolation()
params = model.init(key, a) 
num_params, format_params_fn = get_params_format_fn(params)
print (num_params)

# flatten initial params
params = jax.flatten_util.ravel_pytree(params)[0]  

params_0 = params 


2025-09-08 14:41:42.061663: E external/xla/xla/service/hlo_lexer.cc:438] Failed to parse int literal: 894515288310727292233


12928


In [21]:
@jit
def minibatch(key):
    key1, key2, key3, key4, key5,key6 = key
    batch_pde = random.choice(key1, n_pde , (BS_PDE,),replace = False)
    batch_bc_l = random.choice(key2, n_l_BC, (BS_l_BC,),replace = False)   
    batch_bc_r = random.choice(key3, n_r_BC, (BS_r_BC,),replace = False)   
    batch_bc_tb = random.choice(key4, n_tb_BC, (BS_tb_BC,),replace = False)   
    batch_ad = random.choice(key5, n_ad, (BS_ad,),replace = False)
    batch_X = jnp.vstack([train_X[batch_pde], 
                          data_X_l_BC[batch_bc_l], 
                          data_X_r_BC[batch_bc_r],
                          data_X_bc_tb[batch_bc_tb],
                          ad_points_X[batch_ad],
                          ])
    batch_Y = jnp.vstack([train_Y[batch_pde], 
                          data_Y_l_BC[batch_bc_l],
                          data_Y_r_BC[batch_bc_r], 
                          data_Y_bc_tb[batch_bc_tb],
                          ad_points_Y[batch_ad],
                          ])
    
    return (batch_X, batch_Y)




In [22]:
# loss function
def eval_loss(params,params_0, inputs, labels):
    pred = model.apply(format_params_fn(params), inputs)
    u_pinn, v_pinn, p_pinn, residuals_continuity, residuals_momentum_1, residuals_momentum_2,res_u,res_v, outlet_bc_u,outlet_bc_v,l_bc, r_bc,res_c,res_p = jnp.split(pred, 14, axis=1)
    gt_u_all, gt_v_all,_ = jnp.split(labels, 3, axis=1)
    pred0 = model_0.apply(format_params_fn(params_0), inputs)
    u_0, v_0, p_0,res_u0,res_v0,res_c0,res_p0  = jnp.split(pred0, 7, axis=1)
    pred_interp = model_interp.apply(format_params_fn(params), inputs)
    ad_continuity,ad_momentum_1,ad_momentum_2 = jnp.split(pred_interp, 3, axis=1) 
    
    gt_u = gt_u_all[:BS_PDE]
    gt_v = gt_v_all[:BS_PDE]
    u = u_pinn[:BS_PDE]
    v = v_pinn[:BS_PDE]
    p = p_pinn[:BS_PDE]

    residuals_continuity = residuals_continuity[:BS_PDE]
    residuals_momentum_1 = residuals_momentum_1[:BS_PDE]
    residuals_momentum_2 = residuals_momentum_2[:BS_PDE]
    res_u = res_u[:BS_PDE]
    res_v = res_v[:BS_PDE]
    res_p = res_p[:BS_PDE]
    res_c = res_c[:BS_PDE]
    u_0 = u_0[:BS_PDE]
    v_0 = v_0[:BS_PDE]
    p_0 = p_0[:BS_PDE]
    res_u0 = res_u0[:BS_PDE]
    res_v0 = res_v0[:BS_PDE]
    res_p0 = res_p0[:BS_PDE]
    res_c0 = res_c0[:BS_PDE]


    beta = 0.94   
    

    p_coe = 4/Re
    loss_u = (u - (beta*(-residuals_momentum_1 + res_u0 - res_u)*dx*dx*Re/4) - u_0)
    loss_v = (v - (beta*(-residuals_momentum_2 + res_v0 - res_v)*dx*dx*Re/4) - v_0)
    loss_p = (p - beta*(res_p - res_p0 -p_coe*res_c )/4 - p_0)



    pde_uvp  = jnp.mean(jnp.square(res_c) ) + jnp.mean(jnp.square(residuals_momentum_1)) + jnp.mean(jnp.square(residuals_momentum_2)) 
    uv_rc =  jnp.mean(jnp.abs(loss_u)) + jnp.mean(jnp.abs(loss_v) )+ jnp.mean(jnp.abs(loss_p) )
       
    pde_loss = pde_uvp
    rc_loss = uv_rc
    ad_loss = jnp.mean(jnp.square(ad_continuity)) + jnp.mean(jnp.square(ad_momentum_1)) + jnp.mean(jnp.square(ad_momentum_2))    
            

    bc_outlet_loss = jnp.sum(jnp.square(outlet_bc_u)*r_bc) / r_bc.sum()  + jnp.sum(jnp.square(outlet_bc_v)*r_bc) / r_bc.sum() + jnp.sum(jnp.square(p_pinn)*r_bc) / r_bc.sum()
    
    bc_inlet_loss = jnp.sum(jnp.square(v_pinn)*l_bc) / l_bc.sum() + jnp.sum(jnp.square(u_pinn - gt_u_all)*l_bc) / l_bc.sum()
    bc_wall_loss = jnp.mean(jnp.square(u_pinn[- BS_tb_BC - BS_ad: -BS_ad ])) + jnp.mean(jnp.square(v_pinn[- BS_tb_BC - BS_ad: -BS_ad ]))  # wall BC is the mask for bottom boundary condition
    bc_loss = bc_outlet_loss + bc_inlet_loss + bc_wall_loss 
    
    loss = pde_loss*1 + bc_loss*1 + rc_loss*1  + ad_loss*1 
    
    gt_u = jnp.concatenate([gt_u_all[:BS_PDE], gt_u_all[-BS_ad:]], axis=0)
    gt_v = jnp.concatenate([gt_v_all[:BS_PDE], gt_v_all[-BS_ad:]], axis=0)
    u = jnp.concatenate([u_pinn[:BS_PDE], u_pinn[-BS_ad:]], axis=0)
    v = jnp.concatenate([v_pinn[:BS_PDE], v_pinn[-BS_ad:]], axis=0)
    
    gt_V = jnp.sqrt(gt_u**2 + gt_v**2)  # ground truth velocity magnitude
    V = jnp.sqrt(u**2 + v**2)  # predicted velocity magnitude
    mse_u = jnp.mean(jnp.square(u - gt_u)) 
    mse_v = jnp.mean(jnp.square(v - gt_v)) 
    mse_V = jnp.mean(jnp.square(V - gt_V))  # mean squared error of velocity magnitude
    l2_u = jnp.linalg.norm(u - gt_u) / jnp.linalg.norm(gt_u)
    l2_v = jnp.linalg.norm(v - gt_v) / jnp.linalg.norm(gt_v)
    l2_V = jnp.linalg.norm(V - gt_V) / jnp.linalg.norm(gt_V)  # relative l2 error of velocity magnitude
    
    
    return loss, (mse_u, mse_v, mse_V, l2_u, l2_v, l2_V, pde_loss, bc_loss,rc_loss,ad_loss)

loss_grad = jax.jit(jax.value_and_grad(eval_loss, has_aux=True))    

In [23]:
# weights update  
@jit
def update(params, params_0,opt_state, key):
    batch_X, batch_Y = minibatch(key)
    (loss, (mse_u, mse_v, mse_V, l2_u, l2_v, l2_V, pde_loss, bc_loss,rc_loss,ad_loss)), grad = loss_grad(params,params_0, batch_X, batch_Y)
    updates, opt_state = optimizer.update(grad, opt_state)
    params_0 = params # update u_0

    params = optax.apply_updates(params, updates)
    return params, params_0,opt_state, loss,mse_u, mse_v, mse_V, l2_u, l2_v, l2_V, pde_loss, bc_loss,rc_loss,ad_loss

In [24]:
# optimizer
max_iters = 50000
max_lr = 5e-3
lr_scheduler = optax.warmup_cosine_decay_schedule(init_value=max_lr, peak_value=max_lr, warmup_steps=int(0.0*max_iters),  
                                                  decay_steps=max_iters, end_value=1e-10,exponent=1.0)
optimizer = optax.adam(learning_rate=lr_scheduler) # Choose the method
opt_state = optimizer.init(params)

Training

In [25]:
# training iteration
runtime = 0
train_iters = 0

store = []
while (train_iters <= max_iters):
    # mini-batch update
    start = time.time()
    key1, key2,key3,key4,key5,key6, rng = random.split(rng, 7) # update random generator
    params,params_0, opt_state, loss, mse_u, mse_v, mse_V, l2_u, l2_v, l2_V, pde_loss, bc_loss,rc_loss,ad_loss = update(params,params_0, opt_state, (key1, key2,key3,key4,key5,key6))
    end = time.time()
    runtime += (end-start)    
    # append weights
    if (train_iters % 5000 == 0):
        print ('iter. = %05d,  time = %03ds,  loss = %.2e  |  mse_u = %.2e,  mse_v = %.2e,  mse_V = %.2e,  rl2_u = %.2e,  rl2_v = %.2e,  rl2_V = %.2e, pde =  %.2e, bc = %.2e , rc =  %.2e, adloss = %.2e '%(train_iters, runtime, loss, mse_u, mse_v, mse_V, l2_u, l2_v, l2_V, pde_loss, bc_loss,rc_loss,ad_loss))
        store.append([train_iters, runtime, loss, mse_u, mse_v, mse_V, l2_u, l2_v, l2_V, pde_loss, bc_loss])
    train_iters += 1

store = jnp.array(store)

iter. = 00000,  time = 014s,  loss = 3.43e+01  |  mse_u = 2.71e+00,  mse_v = 1.13e-01,  mse_V = 6.67e-01,  rl2_u = 1.42e+00,  rl2_v = 1.28e+01,  rl2_V = 7.03e-01, pde =  2.36e+01, bc = 2.21e+00 , rc =  2.21e-01, adloss = 8.28e+00 
iter. = 05000,  time = 034s,  loss = 1.66e-02  |  mse_u = 9.32e-02,  mse_v = 2.48e-03,  mse_V = 9.40e-02,  rl2_u = 2.66e-01,  rl2_v = 1.85e+00,  rl2_V = 2.67e-01, pde =  4.68e-03, bc = 5.82e-03 , rc =  3.05e-03, adloss = 3.09e-03 
iter. = 10000,  time = 054s,  loss = 1.41e-02  |  mse_u = 5.63e-02,  mse_v = 1.02e-03,  mse_V = 5.64e-02,  rl2_u = 2.04e-01,  rl2_v = 1.28e+00,  rl2_V = 2.04e-01, pde =  2.13e-03, bc = 4.13e-03 , rc =  2.52e-03, adloss = 5.37e-03 
iter. = 15000,  time = 075s,  loss = 9.90e-03  |  mse_u = 2.05e-02,  mse_v = 2.80e-04,  mse_V = 2.05e-02,  rl2_u = 1.23e-01,  rl2_v = 6.57e-01,  rl2_V = 1.23e-01, pde =  1.17e-03, bc = 1.67e-03 , rc =  2.35e-03, adloss = 4.71e-03 
iter. = 20000,  time = 095s,  loss = 3.65e-03  |  mse_u = 1.05e-02,  mse_v =

In [26]:
sim = pd.read_csv('wavy_channel_361x73_Re100_test.csv',sep=',')
sim_x = sim['x'].values.reshape(-1,1)
sim_y = sim['y'].values.reshape(-1,1)
sim_ls = sim['ls'].values.reshape(-1,1)
sim_u = sim['u'].values.reshape(-1,1)
sim_v = sim['v'].values.reshape(-1,1)
sim_p = sim['p'].values.reshape(-1,1)

data_X, data_Y = np.hstack([sim_x, sim_y,sim_ls]), np.hstack([sim_u, sim_v, sim_p])
test_bool = (data_X[:,-1] <= 0.0)
data_X = data_X[test_bool]
data_Y = data_Y[test_bool]
data_X = data_X[:,:2]  # remove ls


In [27]:
inputs, labels = data_X, data_Y

print(inputs.shape)
uvp = model.apply(format_params_fn(params), inputs)
u, v, p = uvp[:,0:1], uvp[:,1:2], uvp[:,2:3]
gt_u, gt_v,gt_p = jnp.split(labels, 3, axis=-1)

gt_V = jnp.sqrt(gt_u**2 + gt_v**2)  # ground truth velocity magnitude
V = jnp.sqrt(u**2 + v**2)  # predicted velocity magnitude
mse_u = jnp.mean(jnp.square(u - gt_u)) 
mse_v = jnp.mean(jnp.square(v - gt_v)) 
mse_V = jnp.mean(jnp.square(V - gt_V)) 
# mean squared error of velocity magnitude
l2_u = jnp.linalg.norm(u - gt_u) / jnp.linalg.norm(gt_u)
l2_v = jnp.linalg.norm(v - gt_v) / jnp.linalg.norm(gt_v)
l2_V = jnp.linalg.norm(V - gt_V) / jnp.linalg.norm(gt_V)  # relative l2 error of velocity magnitude

mse_p = jnp.mean(jnp.square(p - gt_p))  # mean squared error of pressure
rl2 = jnp.linalg.norm(p - gt_p) / jnp.linalg.norm(gt_p)  # relative l2 error of pressure

print ('[Re=%.1f] :  mse_u = %.2e,  mse_v = %.2e,  mse_V = %.2e,  rl2_u = %.2e,  rl2_v = %.2e,  rl2_V = %.2e, mse_p = %.2e, rl2_p = %.2e'%(Re, mse_u, mse_v, mse_V, l2_u, l2_v, l2_V, mse_p, rl2))


(14480, 2)
[Re=100.0] :  mse_u = 2.03e-05,  mse_v = 3.95e-06,  mse_V = 2.00e-05,  rl2_u = 3.94e-03,  rl2_v = 8.00e-02,  rl2_V = 3.91e-03, mse_p = 6.79e-05, rl2_p = 1.92e-02
